In [1]:
import json
import os
from pathlib import Path
from pprint import pprint
from typing import Optional, Any
from collections import Counter

import torch
import torch.nn as nn
import numpy as np
import wandb
from torch import Tensor
import torch.nn as nn

from datasets import load_from_disk
from transformers import AutoModelForTokenClassification, AutoTokenizer,\
DataCollatorForTokenClassification, TrainingArguments, Trainer, EvalPrediction
import evaluate

from colorama import Fore, Style

In [2]:
from transformers.models.deberta_v2.tokenization_deberta_v2 import DebertaV2Tokenizer
from transformers.models.deberta_v2.modeling_deberta_v2 import DebertaV2ForTokenClassification
from datasets import DatasetDict

In [3]:
model_path = "microsoft/deberta-v3-base"

In [4]:
DATA_DIR = Path(os.getcwd()).parent / "data"
DATA_DIR.mkdir(exist_ok=True)

LABEL_INFO_PATH = DATA_DIR/"label_info.json"
DATASET_PATH = DATA_DIR/"cleaned_ai4privacy_300k_pii"

MODEL_DIR = Path(os.getcwd()).parent / "models"
MODEL_DIR.mkdir(exist_ok=True)

In [5]:
dataset: DatasetDict = load_from_disk(DATASET_PATH) # type:ignore

In [6]:
# label info
with open(LABEL_INFO_PATH) as f:
    label_info: dict = json.load(f)
label2id: dict[str, int] = label_info["label2id"]
id2label = label_info["id2label"]

In [7]:
tokenizer: DebertaV2Tokenizer = AutoTokenizer.from_pretrained(model_path)

model: DebertaV2ForTokenClassification = AutoModelForTokenClassification.from_pretrained(
    model_path, num_labels=len(id2label), id2label=id2label, label2id=label2id)

assert tokenizer.is_fast

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForTokenClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task

In [8]:
for name, param in model.named_parameters():
    if torch.isnan(param).any():
        print(f"NaN in: {name}")

In [9]:
nn.init.normal_(model.classifier.weight, mean=0.0, std=0.02)
nn.init.zeros_(model.classifier.bias)

Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0.], dtype=torch.float16, requires_grad=True)

In [10]:
for name, param in model.named_parameters():
    if torch.isnan(param).any():
        print(f"NaN in: {name}")

In [11]:
for param in model.base_model.parameters():
    param.requires_grad = False
    
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name} requires grad")

classifier.weight requires grad
classifier.bias requires grad


In [12]:
Counter([param.dtype for param in model.parameters()])

Counter({torch.float16: 200})

In [13]:
for name, module in model.named_modules():
    module.to(torch.float32) # type:ignore

In [14]:
Counter([param.dtype for param in model.parameters()])

Counter({torch.float32: 200})

In [15]:
example = dataset["train"][0]

In [16]:
text = "Call John Smith at 555-1234"
enc = tokenizer(text, return_offsets_mapping=True)
for tok, off in zip(enc.tokens(), enc["offset_mapping"]):
    print(f"{tok:20s} {off}  →  '{text[off[0]:off[1]]}'")

[CLS]                (0, 0)  →  ''
▁Call                (0, 4)  →  'Call'
▁John                (4, 9)  →  ' John'
▁Smith               (9, 15)  →  ' Smith'
▁at                  (15, 18)  →  ' at'
▁555                 (18, 22)  →  ' 555'
-                    (22, 23)  →  '-'
1234                 (23, 27)  →  '1234'
[SEP]                (0, 0)  →  ''


In [17]:
def tokenize_and_align_labels_single(example):
    """tokenize and align labels for deberta tokenizer.
    since deberta tokenizer will mark space before as part of the next word,
    masks that start the first letter (not white-space) needs this tokenizer.
    it will first get the word_ids that fall in the mask, then label for
    seqeval

    Args:
        example (_type_): _description_

    Returns:
        _type_: _description_
    """
    encoding = tokenizer(
        example["source_text"],
        return_offsets_mapping=True,
        add_special_tokens=True,
        truncation=True,
        max_length=512
    )
    
    word_ids= encoding.word_ids()
    offsets= encoding["offset_mapping"]
    
    # map offsets to their word_id - for sentence-piece tokenization which 
    # tokenizes the space before the word as part of that word
    word_span: dict[int, tuple[int, int]] = {}
    for offset, word_id in zip(offsets, word_ids):
        if word_id is None:
            continue
        if word_id not in word_span:
            word_span[word_id] = offset
        else:
            prev_start, prev_end = word_span[word_id]
            word_span[word_id] = (
                min(prev_start, offset[0]),
                max(prev_end, offset[1])
            )
    
    # map every word_id to its BI tag
    masks = example["privacy_mask"]
    word_to_ent: dict[int, str] = {}
    for mask in masks:
        words_in_mask = sorted(
            word_id
            for word_id, (w_start, w_end) in word_span.items()
            if w_start < mask["end"] and w_end > mask["start"] 
        )

        for word_id in words_in_mask:
            word_to_ent[word_id] = mask["label"]
    
    # add labels and bio tags
    labels: list[int] = []
    bio_tags: list[Optional[str]] = []
    prev_word_id = None
    prev_ent = None
    for word_id in word_ids:
        if word_id is None:
            labels.append(-100)
            bio_tags.append(None)
            
        elif word_id not in word_to_ent:
            labels.append(label2id["O"])
            bio_tags.append("O")
        else: 
            entity = word_to_ent[word_id] 
            
            if word_id != prev_word_id:
                tag = f"I-{entity}" if prev_ent == entity else f"B-{entity}"
                labels.append(label2id[tag])
            else:
                tag = f"I-{entity}"
                labels.append(-100)
                
            bio_tags.append(tag)
            prev_ent = entity
        prev_word_id = word_id
    
    encoding["ner_tags"] = bio_tags
    encoding["labels"] = labels
    return encoding

In [18]:
dataset = dataset.map(tokenize_and_align_labels_single, batched=False)

In [19]:
example = dataset["train"][0]
example_encodings = tokenizer(
    example["source_text"],
    return_offsets_mapping=True,
    add_special_tokens=True,
)
line1 = ""
line2 = ""
line3 = ""
line4 = ""
for word, tag, label, word_id in zip(example_encodings.tokens(), example["ner_tags"], example["labels"], example_encodings.word_ids()):
    tag = str(tag)
    word_id = str(word_id)
    label = str(label)
    max_length = max(len(word), len(tag), len(word_id), len(label))
    line1 += word + " " * (max_length - len(word) + 1)
    line2 += tag + " " * (max_length - len(tag) + 1)
    line3 += label + " " * (max_length - len(label) + 1)
    line4 += word_id + " " * (max_length - len(word_id) + 1)
    
print(line1)
print(line2)
print(line3)
print(line4)

[CLS] ▁Subject : ▁Group ▁Messaging ▁for ▁Admissions ▁Process ▁Good ▁morning , ▁everyone , ▁I ▁hope ▁this ▁message ▁finds ▁you ▁well .  ▁As ▁we ▁continue ▁our ▁admissions ▁processes ,  ▁I ▁would ▁like ▁to ▁update ▁you ▁on ▁the ▁latest ▁developments ▁and ▁key ▁information .  ▁Please ▁find ▁below ▁the ▁timeline ▁for ▁our ▁upcoming ▁meetings :  ▁- ▁          wyn        q          vr         h          053        ▁- ▁Meeting ▁at ▁10    :      20     am     ▁- ▁l         uka        .          burg       ▁- ▁Meeting ▁at ▁21    ▁- ▁qa        hil        .          witt       auer       ▁- ▁Meeting ▁at ▁quarter ▁past  ▁13    ▁- ▁g         hola       m          hos        s          ein        .          ru         schke      ▁- ▁Meeting ▁at ▁9     :      47     ▁PM    ▁- ▁p         d          m          jr         s          y          oz         14         60         [SEP] 
None  O        O O      O          O    O           O        O     O        O O         O O  O     O     O        O      O

In [20]:
# some tokens include whitespace
print(example["source_text"][310:313].replace(" ", "_"))

_10


In [21]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

seqeval = evaluate.load("seqeval")

In [22]:
from collections import Counter
def compute_metrics(p: EvalPrediction) -> dict[str, float]:
    predictions, label_ids = p.predictions, p.label_ids
    predictions = np.argmax(predictions, axis=-1)
    
    true_preds = [
        [id2label[str(pred)] for pred, tgt_id in zip(preds_row, labels_row) if tgt_id != -100]
        for preds_row, labels_row in zip(predictions, label_ids)
    ]
    
    true_labels =[
        [id2label[str(tgt_id)] for tgt_id in labels_row if tgt_id != -100]
        for labels_row in label_ids
    ]
    
    # TODO: remove
    all_preds = [label for seq in true_preds for label in seq]
    print("Prediction distribution:", Counter(all_preds))
    
    results = seqeval.compute(predictions=true_preds, references=true_labels)
    
    flat = {}
    if results is not None:
        flat.update({
            "precision": results["overall_precision"],
            "recall": results["overall_recall"],
            "f1": results["overall_f1"],
            "accuracy": results["overall_accuracy"],
        })
        
        for entity, scores in results.items():
            if isinstance(scores, dict): # filter scores for individual labels
                flat[f"{entity}_f1"] = scores["f1"]
                flat[f"{entity}_support"] = scores["number"]
    
    return flat

In [23]:
class WeightedTokenClassificationTrainer(Trainer):
    def __init__(self, *args, class_weights: Optional[torch.Tensor]=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        
    def compute_loss(
        self, 
        model: nn.Module, 
        inputs: dict[str, Tensor | Any], 
        return_outputs: bool = False, 
        num_items_in_batch: Tensor | int | None = None
    ) -> Tensor | tuple[Tensor, Any]:
        
        labels = inputs.get("labels")
        outputs = model(**inputs)
        
        if labels is None:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs.loss
            return (loss, outputs) if return_outputs else loss
        
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs.logits
        weight = None
        if self.class_weights is not None:
            weight = self.class_weights.to(logits.device)
            
        loss_fct = torch.nn.CrossEntropyLoss(weight=weight, ignore_index=-100)
        loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


In [24]:
wandb.init(project="pii-redaction", name="frozen-backbone-with-tf32")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/zelluzy/.netrc.
wandb: Currently logged in as: bengid (bengid-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [25]:
for name, param in model.named_parameters():
    param.requires_grad = False if not name.startswith("classifier") else True
    if param.requires_grad:
        print(f"{name} requires grad")

training_args_phase1 = TrainingArguments(
    output_dir=str(MODEL_DIR),
    report_to="wandb",
    learning_rate=1e-3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    warmup_ratio=0.1,
    bf16=False,
    tf32=True,
    max_grad_norm=1.0,
)

trainer = WeightedTokenClassificationTrainer(
    model=model,
    args=training_args_phase1,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


classifier.weight requires grad
classifier.bias requires grad


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,Bod F1,Bod Support,Building F1,Building Support,City F1,City Support,Country F1,Country Support,Date F1,Date Support,Driverlicense F1,Driverlicense Support,Email F1,Email Support,Geocoord F1,Geocoord Support,Givenname1 F1,Givenname1 Support,Givenname2 F1,Givenname2 Support,Idcard F1,Idcard Support,Ip F1,Ip Support,Lastname1 F1,Lastname1 Support,Lastname2 F1,Lastname2 Support,Lastname3 F1,Lastname3 Support,Pass F1,Pass Support,Passport F1,Passport Support,Postcode F1,Postcode Support,Secaddress F1,Secaddress Support,Sex F1,Sex Support,Socialnumber F1,Socialnumber Support,State F1,State Support,Street F1,Street Support,Tel F1,Tel Support,Time F1,Time Support,Title F1,Title Support,Username F1,Username Support
1,0.175023,0.131222,0.751208,0.671808,0.709292,0.965874,0.719733,1124,0.883721,963,0.782523,989,0.694706,757,0.669346,837,0.562092,1142,0.810137,1206,0.554455,104,0.452656,904,0.261398,255,0.626497,1300,0.865169,1028,0.560685,1158,0.226950,313,0.146341,105,0.901186,784,0.659332,1173,0.856524,954,0.681716,440,0.709443,969,0.648394,1285,0.861763,995,0.743891,967,0.706612,991,0.780502,1825,0.624317,906,0.697417,1295
2,0.152137,0.120419,0.766909,0.716904,0.741064,0.969486,0.755245,1124,0.894596,963,0.807748,989,0.784471,757,0.694809,837,0.600489,1142,0.841082,1206,0.582960,104,0.591295,904,0.355191,255,0.645211,1300,0.877901,1028,0.561313,1158,0.297959,313,0.018349,105,0.916070,784,0.679819,1173,0.868530,954,0.737778,440,0.732293,969,0.676433,1285,0.885864,995,0.773631,967,0.759698,991,0.803260,1825,0.680080,906,0.762656,1295


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Prediction distribution: Counter({'O': 294071, 'B-TIME': 1575, 'I-STREET': 1415, 'B-IDCARD': 1299, 'B-PASSPORT': 1235, 'B-EMAIL': 1160, 'B-LASTNAME1': 1134, 'B-USERNAME': 1074, 'B-CITY': 1011, 'B-GIVENNAME1': 960, 'B-IP': 941, 'B-POSTCODE': 927, 'B-STREET': 923, 'B-STATE': 911, 'B-BUILDING': 906, 'B-SOCIALNUMBER': 900, 'B-BOD': 876, 'I-BOD': 855, 'B-TEL': 852, 'B-DRIVERLICENSE': 818, 'B-DATE': 791, 'I-TIME': 780, 'I-TEL': 777, 'B-PASS': 747, 'B-COUNTRY': 674, 'B-SEX': 660, 'I-DATE': 602, 'I-SOCIALNUMBER': 570, 'B-TITLE': 558, 'I-SECADDRESS': 434, 'I-CITY': 418, 'I-DRIVERLICENSE': 410, 'B-SECADDRESS': 366, 'I-COUNTRY': 337, 'I-POSTCODE': 262, 'B-LASTNAME2': 177, 'I-SEX': 155, 'B-GIVENNAME2': 111, 'I-GEOCOORD': 110, 'B-GEOCOORD': 75, 'I-USERNAME': 44, 'I-GIVENNAME1': 39, 'I-PASSPORT': 34, 'I-TITLE': 33, 'I-LASTNAME1': 29, 'I-IDCARD': 19, 'I-PASS': 15, 'I-IP': 13, 'I-STATE': 11, 'B-LASTNAME3': 4, 'I-EMAIL': 2})


TrainOutput(global_step=3740, training_loss=0.24722221292913915, metrics={'train_runtime': 241.1867, 'train_samples_per_second': 248.007, 'train_steps_per_second': 15.507, 'total_flos': 5721316782820488.0, 'train_loss': 0.24722221292913915, 'epoch': 2.0})

In [26]:
for name, param in model.named_parameters():
    if torch.isnan(param).any():
        print(f"NaN in: {name}")

In [27]:
# Unfreeze the backbone
for param in model.base_model.parameters():
    param.requires_grad = True

training_args_phase2 = TrainingArguments(
    output_dir=str(MODEL_DIR),
    learning_rate=2e-5, # low learning_rate
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    warmup_ratio=0.1,
    bf16=False,
    tf32=True,
    max_grad_norm=1.0,
)

trainer = WeightedTokenClassificationTrainer(
    model=model,
    args=training_args_phase2,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,Bod F1,Bod Support,Building F1,Building Support,City F1,City Support,Country F1,Country Support,Date F1,Date Support,Driverlicense F1,Driverlicense Support,Email F1,Email Support,Geocoord F1,Geocoord Support,Givenname1 F1,Givenname1 Support,Givenname2 F1,Givenname2 Support,Idcard F1,Idcard Support,Ip F1,Ip Support,Lastname1 F1,Lastname1 Support,Lastname2 F1,Lastname2 Support,Lastname3 F1,Lastname3 Support,Pass F1,Pass Support,Passport F1,Passport Support,Postcode F1,Postcode Support,Secaddress F1,Secaddress Support,Sex F1,Sex Support,Socialnumber F1,Socialnumber Support,State F1,State Support,Street F1,Street Support,Tel F1,Tel Support,Time F1,Time Support,Title F1,Title Support,Username F1,Username Support
1,0.116686,0.039192,0.921142,0.937543,0.929270,0.990795,0.949138,1124,0.971223,963,0.948504,989,0.958743,757,0.876485,837,0.918620,1142,0.971032,1206,0.936585,104,0.804540,904,0.740406,255,0.914353,1300,0.985075,1028,0.797136,1158,0.673529,313,0.687225,105,0.980940,784,0.924210,1173,0.967876,954,0.947248,440,0.947844,969,0.927742,1285,0.976488,995,0.950485,967,0.936318,991,0.966612,1825,0.935849,906,0.957238,1295
2,0.064615,0.030110,0.930984,0.954701,0.942694,0.992587,0.953037,1124,0.978306,963,0.960000,989,0.956129,757,0.889286,837,0.934717,1142,0.980808,1206,0.947867,104,0.860082,904,0.807143,255,0.936539,1300,0.990790,1028,0.835509,1158,0.747029,313,0.740000,105,0.970960,784,0.945638,1173,0.969979,954,0.953777,440,0.953676,969,0.954615,1285,0.977398,995,0.960121,967,0.945219,991,0.965349,1825,0.967991,906,0.948204,1295
3,0.048085,0.027866,0.938698,0.957003,0.947762,0.993392,0.971581,1124,0.981328,963,0.968579,989,0.957352,757,0.922719,837,0.955437,1142,0.986381,1206,0.956938,104,0.847286,904,0.767213,255,0.931835,1300,0.995612,1028,0.825056,1158,0.714530,313,0.666667,105,0.960699,784,0.951220,1173,0.969168,954,0.968326,440,0.964431,969,0.967258,1285,0.978489,995,0.961013,967,0.968389,991,0.974470,1825,0.970000,906,0.957983,1295
4,0.032436,0.025474,0.950290,0.959344,0.954796,0.994228,0.975610,1124,0.980352,963,0.970090,989,0.968202,757,0.927415,837,0.956825,1142,0.986448,1206,0.970874,104,0.872397,904,0.799283,255,0.944037,1300,0.993680,1028,0.846422,1158,0.761589,313,0.781726,105,0.970923,784,0.957926,1173,0.972807,954,0.971815,440,0.967841,969,0.969032,1285,0.983000,995,0.967709,967,0.966867,991,0.978887,1825,0.973333,906,0.966076,1295
5,0.027594,0.025856,0.951033,0.960556,0.955771,0.994395,0.975588,1124,0.982365,963,0.973922,989,0.968120,757,0.928741,837,0.961016,1142,0.986842,1206,0.970874,104,0.873724,904,0.812386,255,0.945677,1300,0.995127,1028,0.849849,1158,0.760331,313,0.791878,105,0.971609,784,0.958333,1173,0.974120,954,0.973952,440,0.965834,969,0.967540,1285,0.982474,995,0.975258,967,0.965552,991,0.978130,1825,0.970767,906,0.964987,1295


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Prediction distribution: Counter({'O': 290666, 'B-TIME': 1716, 'I-STREET': 1410, 'B-IDCARD': 1224, 'B-SOCIALNUMBER': 1174, 'B-EMAIL': 1153, 'B-USERNAME': 1151, 'B-BOD': 1088, 'B-PASSPORT': 1064, 'B-LASTNAME1': 1060, 'B-DRIVERLICENSE': 1041, 'B-CITY': 1000, 'B-STATE': 1000, 'B-IP': 985, 'B-BUILDING': 962, 'B-STREET': 958, 'B-POSTCODE': 953, 'B-TEL': 935, 'I-TIME': 924, 'B-SEX': 917, 'B-GIVENNAME1': 915, 'I-TEL': 894, 'I-BOD': 873, 'B-TITLE': 849, 'B-DATE': 820, 'B-COUNTRY': 770, 'B-PASS': 766, 'I-SOCIALNUMBER': 621, 'I-DRIVERLICENSE': 588, 'I-DATE': 570, 'I-CITY': 464, 'I-SECADDRESS': 442, 'B-SECADDRESS': 438, 'I-COUNTRY': 323, 'B-GIVENNAME2': 294, 'B-LASTNAME2': 290, 'I-POSTCODE': 250, 'I-SEX': 221, 'I-USERNAME': 174, 'I-LASTNAME1': 166, 'I-PASSPORT': 139, 'I-IDCARD': 120, 'I-TITLE': 102, 'I-EMAIL': 102, 'I-GEOCOORD': 102, 'B-GEOCOORD': 101, 'B-LASTNAME3': 92, 'I-GIVENNAME1': 83, 'I-PASS': 66, 'I-IP': 51, 'I-STATE': 23, 'I-LASTNAME2': 6, 'I-BUILDING': 4})


TrainOutput(global_step=9350, training_loss=0.06486552294562845, metrics={'train_runtime': 1329.5046, 'train_samples_per_second': 112.478, 'train_steps_per_second': 7.033, 'total_flos': 1.3014394085077224e+16, 'train_loss': 0.06486552294562845, 'epoch': 5.0})

In [34]:
for name, param in model.named_parameters():
    if torch.isnan(param).any():
        print(f"NaN in: {name}")
        

In [29]:
example = dataset["train"][0]
example_encodings = tokenizer(
    example["source_text"],
    return_offsets_mapping=True,
    add_special_tokens=True,
)
line1 = ""
line2 = ""
line3 = ""
line4 = ""
for word, tag, label, word_id in zip(example_encodings.tokens(), example["ner_tags"], example["labels"], example_encodings.word_ids()):
    tag = str(tag)
    word_id = str(word_id)
    label = str(label)
    max_length = max(len(word), len(tag), len(word_id), len(label))
    line1 += word + " " * (max_length - len(word) + 1)
    line2 += tag + " " * (max_length - len(tag) + 1)
    line3 += label + " " * (max_length - len(label) + 1)
    line4 += word_id + " " * (max_length - len(word_id) + 1)
    
print(line1)
print(line2)
print(line3)
print(line4)

[CLS] ▁Subject : ▁Group ▁Messaging ▁for ▁Admissions ▁Process ▁Good ▁morning , ▁everyone , ▁I ▁hope ▁this ▁message ▁finds ▁you ▁well .  ▁As ▁we ▁continue ▁our ▁admissions ▁processes ,  ▁I ▁would ▁like ▁to ▁update ▁you ▁on ▁the ▁latest ▁developments ▁and ▁key ▁information .  ▁Please ▁find ▁below ▁the ▁timeline ▁for ▁our ▁upcoming ▁meetings :  ▁- ▁          wyn        q          vr         h          053        ▁- ▁Meeting ▁at ▁10    :      20     am     ▁- ▁l         uka        .          burg       ▁- ▁Meeting ▁at ▁21    ▁- ▁qa        hil        .          witt       auer       ▁- ▁Meeting ▁at ▁quarter ▁past  ▁13    ▁- ▁g         hola       m          hos        s          ein        .          ru         schke      ▁- ▁Meeting ▁at ▁9     :      47     ▁PM    ▁- ▁p         d          m          jr         s          y          oz         14         60         [SEP] 
None  O        O O      O          O    O           O        O     O        O O         O O  O     O     O        O      O

In [30]:
Counter([param.dtype for param in model.parameters()])

Counter({torch.float32: 200})

In [31]:
import html as html_lib

def log_pii_test_results(
    trainer,
    test_dataset,
    tokenizer,
    table_name: str = "pii_test_results",
    run: Optional[wandb.sdk.wandb_run.Run] = None,
) -> wandb.Table:
    id2label = trainer.model.config.id2label
    pred_output = trainer.predict(test_dataset)
    pred_ids = np.argmax(pred_output.predictions, axis=-1)
    true_ids = pred_output.label_ids

    table = wandb.Table(columns=["id", "annotated_sequence", "f1", "precision", "recall", "tp", "fp", "fn"])

    for i in range(len(test_dataset)):
        tokens, true_labels, pred_labels = [], [], []
        for token_id, true_id, pred_id in zip(test_dataset[i]["input_ids"], true_ids[i], pred_ids[i]):
            if true_id == -100:
                continue
            tokens.append(tokenizer.convert_ids_to_tokens(token_id))
            true_labels.append(id2label[str(true_id)])
            pred_labels.append(id2label[str(pred_id)])

        tp = fp = fn = 0
        for true, pred in zip(true_labels, pred_labels):
            if true != "O" and pred == true:  tp += 1
            elif true != "O" and pred != true: fn += 1
            elif true == "O" and pred != "O":  fp += 1

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

        parts = []
        for token, true, pred in zip(tokens, true_labels, pred_labels):
            t = html_lib.escape(token)
            if true != "O" and pred == true:
                badge = f'<sup style="font-size:9px;padding:1px 4px;border-radius:3px;background:#C0DD97;color:#27500A">{html_lib.escape(true)}</sup>'
                parts.append(f'<span style="background:#C0DD97;color:#27500A;font-weight:500;padding:2px 4px;border-radius:3px;margin:1px 2px;display:inline-block">{t}{badge}</span>')
            elif true != "O" and pred != true:
                label = html_lib.escape(f"{true}→{pred}")
                badge = f'<sup style="font-size:9px;padding:1px 4px;border-radius:3px;background:#F7C1C1;color:#791F1F">{label}</sup>'
                parts.append(f'<span style="background:#F7C1C1;color:#791F1F;font-weight:500;padding:2px 4px;border-radius:3px;margin:1px 2px;display:inline-block">{t}{badge}</span>')
            elif true == "O" and pred != "O":
                badge = f'<sup style="font-size:9px;padding:1px 4px;border-radius:3px;background:#FAC775;color:#633806">{html_lib.escape(pred)}</sup>'
                parts.append(f'<span style="background:#FAC775;color:#633806;font-weight:500;padding:2px 4px;border-radius:3px;margin:1px 2px;display:inline-block">{t}{badge}</span>')
            else:
                parts.append(f'<span style="margin:1px 2px;display:inline-block">{t}</span>')

        seq_html = '<div style="font-family:monospace;font-size:12px;line-height:2">' + "".join(parts) + "</div>"
        table.add_data(i, wandb.Html(seq_html), round(f1, 4), round(precision, 4), round(recall, 4), tp, fp, fn)

    (run or wandb).log({table_name: table})
    return table

In [32]:
log_pii_test_results(trainer, dataset["test"], tokenizer)

Prediction distribution: Counter({'O': 287761, 'B-TIME': 1754, 'I-STREET': 1303, 'B-USERNAME': 1220, 'B-EMAIL': 1174, 'B-IDCARD': 1170, 'B-PASSPORT': 1123, 'B-LASTNAME1': 1108, 'B-SOCIALNUMBER': 1108, 'B-BOD': 1100, 'B-DRIVERLICENSE': 1071, 'B-IP': 1050, 'B-CITY': 997, 'B-STATE': 980, 'B-GIVENNAME1': 971, 'B-SEX': 962, 'B-STREET': 944, 'I-TIME': 936, 'B-POSTCODE': 933, 'B-BUILDING': 931, 'B-TEL': 900, 'B-TITLE': 876, 'B-DATE': 856, 'I-TEL': 833, 'I-BOD': 798, 'B-COUNTRY': 788, 'B-PASS': 779, 'I-DATE': 625, 'I-SOCIALNUMBER': 565, 'I-DRIVERLICENSE': 527, 'I-CITY': 471, 'B-SECADDRESS': 420, 'I-SECADDRESS': 417, 'I-POSTCODE': 384, 'I-COUNTRY': 334, 'B-LASTNAME2': 288, 'B-GIVENNAME2': 252, 'I-USERNAME': 221, 'I-SEX': 207, 'I-LASTNAME1': 172, 'I-IDCARD': 138, 'I-EMAIL': 134, 'I-GIVENNAME1': 126, 'I-TITLE': 124, 'B-GEOCOORD': 111, 'I-GEOCOORD': 111, 'I-PASSPORT': 104, 'B-LASTNAME3': 93, 'I-PASS': 93, 'I-IP': 58, 'I-STATE': 14, 'I-LASTNAME2': 6, 'I-BUILDING': 1})


In [33]:
wandb.finish()

eval/BOD_f1,▁█
eval/BOD_support,▁▁
eval/BUILDING_f1,▁█
eval/BUILDING_support,▁▁
eval/CITY_f1,▁█
eval/CITY_support,▁▁
eval/COUNTRY_f1,▁█
eval/COUNTRY_support,▁▁
eval/DATE_f1,▁█
eval/DATE_support,▁▁
+57,...
